[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/Time-Series-Analysis/blob/main/Quantlets/Ch_09/TSA_ch9_regularisation_trees/TSA_ch9_regularisation_trees.ipynb)

# TSA_ch9_regularisation_trees

Ridge and lasso coefficient paths on the 30 load features; a depth-2 regression tree for tomorrow's load; a single tree, a random forest and histogram gradient boosting (two learning rates) on validation data; trees cannot extrapolate: random forests on the level and on the monthly changes of the Romanian HICP index after 2020.

*Time Series Analysis (TSA), Chapter 9: Machine learning for time series. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/TSA/Chapter_9'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '1'   # small data: one thread per model fit is faster
# PyTorch is preinstalled in Google Colab; it is needed only for the LSTM

In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from scipy import optimize
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Chart style

In [ ]:
# Palette (RGB values of latex/preamble.tex)
MainBlue = '#1A3A6E'
IDAred = '#CD0000'
Forest = '#2E7D32'
Amber = '#B5853F'
Orange = '#E67E22'
Purple = '#8E44AD'
Teal = '#17A2B8'
Crimson = '#DC3545'
DarkText = '#1F2A44'          # text, axes and ticks (dark navy, not grey)

PALETTE = [MainBlue, IDAred, Forest, Amber, Purple, Orange, Teal, Crimson]
# fixed colours for the series used in several chapters
COL = {'sp500': MainBlue, 'bet': IDAred, 'bettr': Orange, 'dax': Teal, 'btc': Amber, 'eth': Crimson,
       'eurron': Forest, 'gold': Purple, 'vix': Crimson, 'stoxx50': Forest, 'ndx': Teal}


def apply():
    """Set the course style for matplotlib."""
    rc = plt.rcParams
    rc['figure.facecolor'] = 'none'
    rc['axes.facecolor'] = 'none'
    rc['savefig.facecolor'] = 'none'
    rc['savefig.transparent'] = True
    rc['axes.grid'] = False
    rc['font.family'] = 'sans-serif'
    rc['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
    # font sizes for slides (charts are 9-11 inches wide and shown at 8-14 cm)
    rc['font.size'] = 12
    rc['axes.labelsize'] = 13
    rc['axes.titlesize'] = 13
    rc['xtick.labelsize'] = 11.5
    rc['ytick.labelsize'] = 11.5
    rc['legend.fontsize'] = 11
    rc['axes.spines.top'] = False
    rc['axes.spines.right'] = False
    rc['axes.linewidth'] = 0.6
    rc['lines.linewidth'] = 1.4
    rc['legend.facecolor'] = 'none'
    rc['legend.framealpha'] = 0
    rc['legend.frameon'] = False
    for k in ('text.color', 'axes.labelcolor', 'axes.edgecolor', 'xtick.color', 'ytick.color', 'axes.titlecolor'):
        rc[k] = DarkText
    rc['axes.prop_cycle'] = mpl.cycler(color=PALETTE)


def legend_outside_bottom(ax, ncol=2, y=-0.22, **kw):
    """Place the legend outside the plot, bottom centre (for a figure with several axes, pass the last one
    or use fig.legend with the same arguments)."""
    return ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False, **kw)


def fig_legend_bottom(fig, handles=None, labels=None, ncol=3, y=-0.02):
    """One legend for a whole figure (several panels), below the panels."""
    if handles is None:
        handles, labels = [], []
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in labels and not l.startswith('_'):
                    handles.append(h)
                    labels.append(l)
    return fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def _is_grey(c, tol=0.06):
    try:
        r, g, b = to_rgb(c)
    except ValueError:
        return False
    return max(r, g, b) - min(r, g, b) < tol and 0.25 < (r + g + b) / 3 < 0.95


def check_no_grey(fig):
    """House rule: no grey series and no grey text. Raises ValueError listing the offending elements."""
    bad = []
    for ax in fig.axes:
        for ln in ax.get_lines():
            if not ln.get_label().startswith('_') and _is_grey(ln.get_color()):
                bad.append(f'line {ln.get_label()!r}')
        for coll in ax.collections:
            fc = coll.get_facecolor()
            if len(fc) and coll.get_label() and not coll.get_label().startswith('_') and _is_grey(fc[0][:3]):
                bad.append(f'series {coll.get_label()!r}')
        for t in ax.texts + [ax.title, ax.xaxis.label, ax.yaxis.label]:
            if t.get_text() and _is_grey(t.get_color()):
                bad.append(f'text {t.get_text()[:30]!r}')
    if bad:
        raise ValueError('grey elements: ' + ', '.join(bad))


def save_fig(name, out_dir=None, show=True):
    """Save the figure as transparent PDF and PNG (OUT_DIR if defined, else the current folder), then show it."""
    d = out_dir or globals().get('OUT_DIR', '.')
    plt.savefig(os.path.join(d, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(d, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    if show:
        plt.show()
    plt.close()


apply()
# the chapter scripts call the style as st.<name> (import tsa_style as st): the same names here
import types
st = types.SimpleNamespace(COL=COL, PALETTE=PALETTE, MainBlue=MainBlue, IDAred=IDAred, Forest=Forest, Amber=Amber,
                           Orange=Orange, Purple=Purple, Teal=Teal, Crimson=Crimson, DarkText=DarkText, apply=apply, legend_outside_bottom=legend_outside_bottom,
                           fig_legend_bottom=fig_legend_bottom, save_fig=save_fig, check_no_grey=check_no_grey)

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/Time-Series-Analysis/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'          # last day of the saved data

# name -> (symbol, label, group, field, default start)
SERIES = {
    # equity indices
    'sp500':    ('GSPC.INDX',     'S&P 500',               'Equity', 'close', '2000-01-01'),
    'ndx':      ('NDX.INDX',      'Nasdaq 100',            'Equity', 'close', '2000-01-01'),
    'dax':      ('GDAXI.INDX',    'DAX',                   'Equity', 'close', '2000-01-01'),
    'stoxx50':  ('STOXX50E.INDX', 'Euro Stoxx 50',         'Equity', 'close', '2000-01-01'),
    'nikkei':   ('N225.INDX',     'Nikkei 225',            'Equity', 'close', '2000-01-01'),
    'bet':      ('BET',           'BET',                   'Equity', 'close', '2000-01-01'),
    'bettr':    ('BETTR.INDX',    'BET-TR',                'Equity', 'close', '2014-09-23'),
    'betxt':    ('BETXT.INDX',    'BET-XT',                'Equity', 'close', '2011-08-12'),
    'betfi':    ('BETFI.INDX',    'BET-FI',                'Equity', 'close', '2012-01-25'),
    'wig20':    ('WIG20.INDX',    'WIG20',                 'Equity', 'close', '2000-01-01'),
    'bux':      ('BUX.INDX',      'BUX',                   'Equity', 'close', '2000-01-01'),
    'px':       ('PX.INDX',       'PX',                    'Equity', 'close', '2000-01-01'),
    'vix':      ('VIX.INDX',      'VIX',                   'Volatility', 'close', '2000-01-01'),
    # ETFs (adjusted close)
    'spy':      ('SPY.US',        'SPY',                   'ETF', 'adjusted_close', '2000-01-01'),
    'tlt':      ('TLT.US',        'TLT (US Treasuries 20y+)', 'ETF', 'adjusted_close', '2002-07-30'),
    'gld':      ('GLD.US',        'GLD (gold ETF)',        'ETF', 'adjusted_close', '2004-11-18'),
    'tvbetetf': ('TVBETETF.RO',   'TVBETETF',              'ETF', 'adjusted_close', '2015-06-12'),
    # Bucharest Stock Exchange (adjusted close)
    'tlv':      ('TLV.RO',        'Banca Transilvania',    'Stock', 'adjusted_close', '2000-01-01'),
    'snp':      ('SNP.RO',        'OMV Petrom',            'Stock', 'adjusted_close', '2001-09-04'),
    'brd':      ('BRD.RO',        'BRD',                   'Stock', 'adjusted_close', '2001-01-16'),
    'tgn':      ('TGN.RO',        'Transgaz',              'Stock', 'adjusted_close', '2008-01-25'),
    'sng':      ('SNG.RO',        'Romgaz',                'Stock', 'adjusted_close', '2013-11-12'),
    'snn':      ('SNN.RO',        'Nuclearelectrica',      'Stock', 'adjusted_close', '2013-11-04'),
    'h2o':      ('H2O.RO',        'Hidroelectrica',        'Stock', 'adjusted_close', '2023-07-12'),
    # US stocks (adjusted close)
    'aapl':     ('AAPL.US',       'Apple',                 'Stock', 'adjusted_close', '2000-01-01'),
    'msft':     ('MSFT.US',       'Microsoft',             'Stock', 'adjusted_close', '2000-01-01'),
    'nvda':     ('NVDA.US',       'NVIDIA',                'Stock', 'adjusted_close', '2000-01-01'),
    'jpm':      ('JPM.US',        'JPMorgan Chase',        'Stock', 'adjusted_close', '2000-01-01'),
    # crypto (close, 7 days a week)
    'btc':      ('BTC-USD.CC',    'Bitcoin',               'Crypto', 'close', '2014-09-17'),
    'eth':      ('ETH-USD.CC',    'Ethereum',              'Crypto', 'close', '2015-08-07'),
    'sol':      ('SOL-USD.CC',    'Solana',                'Crypto', 'close', '2020-04-11'),
    'usdt':     ('USDT-USD.CC',   'Tether (USDT)',         'Crypto', 'close', '2015-02-26'),
    # FX and commodities
    'eurron':   ('REF:EUR',       'EUR/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'usdron':   ('REF:USD',       'USD/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'eurron_eodhd': ('EURRON.FOREX', 'EUR/RON (EODHD)',    'FX', 'close', '2005-07-01'),
    'eurusd':   ('EURUSD.FOREX',  'EUR/USD',               'FX', 'close', '2002-05-06'),
    'gold':     ('XAUUSD.FOREX',  'Gold (XAU/USD)',        'Commodity', 'close', '2000-01-01'),
    # government bond yields (close, % p.a.)
    'us10y':    ('US10Y.GBOND',   'US 10Y yield',          'Yield', 'close', '2000-01-01'),
    'de10y':    ('DE10Y.GBOND',   'Germany 10Y yield',     'Yield', 'close', '2007-03-28'),
    'ro10y':    ('RO10Y.GBOND',   'Romania 10Y yield',     'Yield', 'close', '2007-08-17'),
}
LABELS = {k: v[1] for k, v in SERIES.items()}
_CACHE = {}


def read_market(symbol):
    """Daily table of one symbol: local copy of the course data, otherwise the TSA repository on GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname) if MARKET_DIR else ''
    src = path if path and os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official BNR reference rate (RON per unit of currency), from the public yearly XML archives."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}"(?: multiplier="\d+")?>([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END, field=None):
    """Daily closing price of a named series, cleaned with the course conventions."""
    symbol, label, group, field0, start0 = SERIES[name]
    start, field = start or start0, field or field0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    t = read_market(symbol)
    s = (t[field] if field in t.columns else t['close']).loc[start:end]
    s = pd.to_numeric(s, errors='coerce').dropna()
    if group != 'Yield':
        s = s[s > 0]
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    s.index.name = 'date'
    return s.rename(name)


def load_ohlc(name, start=None, end=END):
    """Open, high, low, close (and volume) of a series with OHLC data (range-based volatility estimators)."""
    symbol, _, group, _, start0 = SERIES[name]
    t = read_market(symbol).loc[start or start0:end]
    cols = [c for c in ('open', 'high', 'low', 'close', 'volume') if c in t.columns]
    t = t[cols].dropna()
    if group != 'Crypto':
        t = t[t.index.dayofweek < 5]
    return t[(t[['open', 'high', 'low', 'close']] > 0).all(axis=1)] if 'open' in cols else t


def log_returns(name, start=None, end=END, pct=True):
    """Daily log returns r_t = ln P_t - ln P_{t-1} (in % by default), on the series' own calendar."""
    r = np.log(load_close(name, start, end)).diff().dropna()
    return (100 * r if pct else r).rename(name)


def simple_returns(name, start=None, end=END, pct=True):
    """Daily simple returns R_t = P_t / P_{t-1} - 1 (in % by default)."""
    r = load_close(name, start, end).pct_change().dropna()
    return (100 * r if pct else r).rename(name)


def load_panel(names, start=None, end=END, kind='close'):
    """Several series aligned on date: kind = 'close' or 'returns' (NaN where a market does not trade)."""
    f = load_close if kind == 'close' else log_returns
    return pd.concat([f(n, start, end) for n in names], axis=1)


def periods_per_year(r):
    """Average number of observations per calendar year (the actual frequency of the series)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years


def read_fred(ids, start=None, end=None):
    """FRED series (St. Louis Fed) from the public fredgraph CSV, e.g. read_fred('UNRATE') or
    read_fred(['GDPC1', 'CPIAUCSL']). Returns a Series (one id) or a DataFrame (several ids)."""
    one = isinstance(ids, str)
    ids = [ids] if one else list(ids)
    out = []
    for sid in ids:
        key = ('fred', sid)
        if key not in _CACHE:
            url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={sid}'
            t = pd.read_csv(url, index_col=0, parse_dates=True, na_values='.')
            _CACHE[key] = pd.to_numeric(t.iloc[:, 0], errors='coerce').rename(sid)
        out.append(_CACHE[key])
    df = pd.concat(out, axis=1).loc[start:end]
    df.index.name = 'date'
    return df.iloc[:, 0].dropna() if one else df


def _eurostat_period(p):
    p = str(p)
    if '-Q' in p:
        y, q = p.split('-Q')
        return pd.Timestamp(int(y), 3 * int(q) - 2, 1)
    if '-' in p and len(p) == 7:
        return pd.Timestamp(p + '-01')
    if len(p) == 4:
        return pd.Timestamp(int(p), 1, 1)
    return pd.Timestamp(p)


def read_eurostat(dataset, key, start=None):
    """One Eurostat series from the public SDMX 2.1 API (SDMX-CSV), e.g. Romanian quarterly real GDP,
    seasonally and calendar adjusted, chain-linked volumes (2010) in million EUR:
        read_eurostat('namq_10_gdp', 'Q.CLV10_MEUR.SCA.B1GQ.RO')
    `key` is the dot-separated series key of the dataset (dimensions in the order of the dataset)."""
    url = (f'https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/{dataset}/{key}?format=SDMX-CSV'
           + (f'&startPeriod={start}' if start else ''))
    ck = ('eurostat', url)
    if ck not in _CACHE:
        t = pd.read_csv(url)
        s = pd.Series(pd.to_numeric(t['OBS_VALUE'], errors='coerce').values,
                      index=[_eurostat_period(p) for p in t['TIME_PERIOD']], name=f'{dataset}:{key}')
        s.index.name = 'date'
        _CACHE[ck] = s.sort_index().dropna()
    return _CACHE[ck]


def load_statsmodels(name):
    """Classic data sets shipped with statsmodels (offline): 'sunspots' (yearly, 1700-2008), 'co2' (weekly,
    Mauna Loa, 1958-2001), 'macrodata' (US quarterly macro, 1959-2009), 'nile' (yearly Nile flow, 1871-1970)."""
    import statsmodels.api as sm
    if name == 'sunspots':
        d = sm.datasets.sunspots.load_pandas().data
        s = d.set_index(pd.to_datetime(d['YEAR'].astype(int).astype(str)))['SUNACTIVITY']
        s.index.name = 'date'
        return s.rename('sunspots')
    if name == 'co2':
        return sm.datasets.co2.load_pandas().data['co2'].rename('co2')
    if name == 'macrodata':
        d = sm.datasets.macrodata.load_pandas().data
        d.index = pd.period_range('1959Q1', periods=len(d), freq='Q').to_timestamp()
        d.index.name = 'date'
        return d
    if name == 'nile':
        d = sm.datasets.nile.load_pandas().data
        s = d.set_index(pd.to_datetime(d['year'].astype(int).astype(str)))['volume']
        s.index.name = 'date'
        return s.rename('nile')
    raise KeyError(f'unknown statsmodels data set: {name}')

## Analysis

In [ ]:
import math

from matplotlib.patches import Rectangle, Circle, FancyArrowPatch

from sklearn.ensemble import (HistGradientBoostingClassifier, HistGradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor)

from sklearn.inspection import permutation_importance

from sklearn.linear_model import LassoCV, LinearRegression, LogisticRegression, RidgeCV, lasso_path

from sklearn.metrics import roc_auc_score

from sklearn.model_selection import KFold, TimeSeriesSplit

from sklearn.neural_network import MLPRegressor

from sklearn.pipeline import make_pipeline

from sklearn.preprocessing import StandardScaler

from sklearn.tree import DecisionTreeRegressor, plot_tree

from dateutil.easter import easter, EASTER_ORTHODOX

HERE = "."

SEED = 2026
# daily load: the file and the evaluation design of Chapter 4 (26 origins, every 14 days, horizon 14 days)
LOAD_FILE = 'ch4_ro_load_hourly.csv'
LOAD_RAW = 'https://raw.githubusercontent.com/danpele/Time-Series-Analysis/main/Quantlets/Ch_04/'
LOAD_END = '2026-06-30'
CV_FIRST, CV_STEP, CV_H = '2025-06-30', 14, 14
LAGS = 14                      # y_t, ..., y_{t-13} at the forecast origin
HICP = ('prc_hicp_minr', 'M.I15.TOTAL.{}')
EU27 = ['AT', 'BE', 'BG', 'CY', 'CZ', 'DE', 'DK', 'EE', 'EL', 'ES', 'FI', 'FR', 'HR', 'HU', 'IE', 'IT', 'LT', 'LU', 'LV',
        'MT', 'NL', 'PL', 'PT', 'RO', 'SE', 'SI', 'SK']
INF_START, INF_OOS = '2005-01-01', 2016          # inflation targeting in Romania since August 2005; tests from 2016
INF_H = [1, 3, 6, 12]
RV_H = 5                                          # volatility target: mean variance proxy over the next 5 days
RV_OOS = {'sp500': '2013-01-01', 'dax': '2013-01-01'}
OHLC_START = {'sp500': '2008-01-01', 'dax': '2008-01-01'}
HAR = ['d', 'w', 'm']
EXT = HAR + ['d1', 'd2', 'd3', 'd4', 'q', 'r', 'r_neg', 'abs_r']
SIGN_FEATS = ['r0', 'r1', 'r2', 'r3', 'r4', 'm5', 'm21', 'm63', 'v21', 'v63']
NAME = {'sp500': 'S&P 500', 'dax': 'DAX', 'bet': 'BET'}
MCOL = {'Seasonal naive': st.Amber, 'ETS': st.Teal, 'SARIMA': st.Purple, 'DHR': st.MainBlue, 'Combination': st.Crimson,
        'Ridge': st.Teal, 'RF': st.Forest, 'GB direct': st.IDAred, 'GB recursive': st.Orange, 'MLP': '#6B8E23',
        'LSTM': st.Purple, 'RW': st.Amber, 'AR': st.MainBlue, 'Lasso': st.Teal, 'GB local': st.IDAred,
        'GB global': st.Orange, 'HAR': st.MainBlue, 'GB': st.IDAred, 'Mean': st.Amber, 'Logit': st.MainBlue}
GB_PARAMS = dict(max_iter=300, learning_rate=0.05, max_leaf_nodes=15, min_samples_leaf=20, l2_regularization=1.0)

FEATS = ['lag0', 'lag1', 'lag2', 'lag3', 'lag4', 'lag5', 'lag6', 'lag7', 'lag8', 'lag9', 'lag10', 'lag11', 'lag12', 'lag13', 'mean7', 'mean28', 'same_dow', 'mon', 'tue', 'wed', 'thu', 'fri', 'sat', 'sin1', 'cos1', 'sin2', 'cos2', 'easter', 'xmas', 'holiday']

INF_FEATS = ['pi0', 'pi1', 'pi2', 'pi3', 'pi6', 'pi12', 'm0', 'm1', 'm2', 'm3s', 'm6s', 'month']

M4_FILE = 'ch9_m4_owa.csv'

HIGHLIGHT = {'lag0': st.MainBlue, 'same_dow': st.IDAred, 'mean7': st.Forest, 'sat': st.Orange, 'holiday': st.Purple, 'easter': st.Amber, 'xmas': st.Teal}

M4_TYPES = {'Statistical': st.MainBlue, 'Combination (S)': st.Teal, 'Combination (S & ML)': st.Purple, 'Combination (ML)': st.Orange, 'Machine Learning': st.IDAred, 'Hybrid': st.Forest, 'Other': st.Amber}

GKX_MODELS = ['OLS-3+H', 'PLS', 'PCR', 'ENet+H', 'GLM+H', 'RF', 'GBRT+H', 'NN1', 'NN2', 'NN3', 'NN4', 'NN5']

GKX_R2 = [0.16, 0.27, 0.26, 0.11, 0.19, 0.33, 0.34, 0.33, 0.39, 0.4, 0.39, 0.36]

GKX_SR = [0.61, 0.72, 0.88, 0.39, 0.76, 0.98, 0.81, 1.17, 1.16, 1.2, 1.35, 1.15]

def ro_holidays(years):
    """Romanian public holidays (Orthodox Easter and Pentecost from dateutil): date -> name (as in Chapter 4)."""
    out = {}
    for y in years:
        e = pd.Timestamp(easter(y, EASTER_ORTHODOX))
        day = pd.Timedelta(days=1)
        hol = {'New Year': [f'{y}-01-01', f'{y}-01-02'], 'Union Day': [f'{y}-01-24'],
               'Easter': [e - 2 * day, e, e + day], 'Labour Day': [f'{y}-05-01'], "Children's Day": [f'{y}-06-01'],
               'Pentecost': [e + 49 * day, e + 50 * day], 'Assumption': [f'{y}-08-15'], "St Andrew's Day": [f'{y}-11-30'],
               'National Day': [f'{y}-12-01'], 'Christmas': [f'{y}-12-25', f'{y}-12-26']}
        if y >= 2024:
            hol['Epiphany'] = [f'{y}-01-06', f'{y}-01-07']
        for k, v in hol.items():
            for x in v:
                out[pd.Timestamp(x)] = k
    return pd.Series(out, name='holiday').sort_index()


def load_daily(end=LOAD_END):
    """Daily mean electricity load of Romania in GW (ENTSO-E hourly values, Chapter 4 extract, Romanian winter time;
    isolated one-hour glitches and missing hours interpolated), 1 January 2022 - 30 June 2026."""
    local = [os.path.join(HERE, '..', 'Ch_04', LOAD_FILE)] + [os.path.join(d, 'Quantlets', 'Ch_04', LOAD_FILE)
                                                              for d in ('.', '..', '../..', '../../..')]
    src = next((p for p in local if os.path.exists(p)), LOAD_RAW + LOAD_FILE)
    s = pd.read_csv(src, index_col=0, parse_dates=True).iloc[:, 0]
    s = s.loc[:pd.Timestamp(end) + pd.Timedelta(hours=21)]
    s.index = s.index + pd.Timedelta(hours=2)
    s = s.reindex(pd.date_range(s.index[0], s.index[-1], freq='h'))
    r = s / ((s.shift(1) + s.shift(-1)) / 2)
    s[(r < 0.7) | (r > 1.3)] = np.nan
    d = s.interpolate().resample('D').mean() / 1000
    return d.loc['2022-01-01':end].rename('load')


def calendar(idx):
    """Calendar features of the dates idx: weekday dummies (Sunday is the base), two annual Fourier pairs, Orthodox
    Easter (Friday to Monday), the Christmas - New Year days (24 December - 2 January) and other public holidays."""
    idx = pd.DatetimeIndex(idx)
    X = pd.DataFrame(index=idx)
    for j, nm in enumerate(['mon', 'tue', 'wed', 'thu', 'fri', 'sat']):
        X[nm] = (idx.dayofweek == j).astype(float)
    doy = idx.dayofyear.values
    for k in (1, 2):
        X[f'sin{k}'] = np.sin(2 * np.pi * k * doy / 365.25)
        X[f'cos{k}'] = np.cos(2 * np.pi * k * doy / 365.25)
    hol = ro_holidays(range(idx[0].year, idx[-1].year + 1)).reindex(idx)
    X['easter'] = (hol == 'Easter').astype(float).values
    X['xmas'] = (((idx.month == 12) & (idx.day >= 24)) | ((idx.month == 1) & (idx.day <= 2))).astype(float)
    X['holiday'] = (hol.notna().values & (X['easter'] == 0).values & (X['xmas'] == 0).values).astype(float)
    return X


def direct_frame(y, h):
    """Supervised table for horizon h: one row per forecast origin t. Features known at t: y_t, ..., y_{t-13}, the
    7- and 28-day means, the last value of the same weekday as the target (y_{t+h-7k}, k = ceil(h/7)) and the calendar
    of the target day t+h. Target: y_{t+h}."""
    X = pd.DataFrame({f'lag{j}': y.shift(j) for j in range(LAGS)}, index=y.index)
    X['mean7'] = y.rolling(7).mean()
    X['mean28'] = y.rolling(28).mean()
    k = math.ceil(h / 7)
    X['same_dow'] = y.shift(7 * k - h)
    cal = calendar(y.index + pd.Timedelta(days=h))
    cal.index = y.index
    X = pd.concat([X, cal], axis=1)
    X['y'] = y.shift(-h)
    return X.iloc[27:]


def origins(y=None, first=CV_FIRST, step=CV_STEP, h=CV_H):
    """The forecast origins of Chapter 4: every 14 days from 30 June 2025, while 14 days of data remain."""
    y = load_daily() if y is None else y
    return list(pd.date_range(first, y.index[-1] - pd.Timedelta(days=h), freq=f'{step}D'))


def oos_r2(y, yhat, ybar):
    """Out-of-sample R^2 against a benchmark forecast ybar: 1 - sum (y - yhat)^2 / sum (y - ybar)^2."""
    y, yhat, ybar = (np.asarray(a, float) for a in (y, yhat, ybar))
    return float(1 - np.sum((y - yhat) ** 2) / np.sum((y - ybar) ** 2))


def dm_raw(l1, l2, h=1):
    """Diebold-Mariano test on two loss series (Chapter 4): HLN small-sample correction, t(n - 1) p-value.
    d_t = l1_t - l2_t: a negative mean favours the first forecast."""
    d = np.asarray(l1, float) - np.asarray(l2, float)
    n = len(d)
    dbar = d.mean()
    g = [np.mean((d[k:] - dbar) * (d[:n - k] - dbar)) for k in range(h)]
    v = (g[0] + 2 * sum(g[1:])) / n
    dm = dbar / np.sqrt(v)
    hln = np.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n) * dm
    return {'dbar': float(dbar), 'dm': float(dm), 'hln': float(hln), 'p': float(2 * stats.t.sf(abs(hln), n - 1)), 'n': int(n)}


def shrinkage_paths(y=None, end=CV_FIRST):
    """Ridge and lasso coefficient paths of the standardised load features (h = 1), data up to the first origin."""
    y = load_daily() if y is None else y
    F = direct_frame(y.loc[:end], 1).dropna()
    Z = StandardScaler().fit_transform(F[FEATS].values)
    t = F['y'].values - F['y'].mean()
    alphas_l, coefs_l, _ = lasso_path(Z, t, n_alphas=60, eps=1e-4)
    lams = np.logspace(-2, 5, 60)
    coefs_r = np.array([np.linalg.solve(Z.T @ Z + l * np.eye(Z.shape[1]), Z.T @ t) for l in lams]).T
    cv = LassoCV(cv=TimeSeriesSplit(5), n_alphas=60, eps=1e-4).fit(Z, t)
    kept = [FEATS[j] for j in np.flatnonzero(np.abs(cv.coef_) > 1e-8)]
    order = np.argsort(-np.abs(cv.coef_))
    top = [(FEATS[j], float(cv.coef_[j])) for j in order[:6]]
    entry = {}
    for j, f in enumerate(FEATS):
        nz = np.flatnonzero(np.abs(coefs_l[j]) > 1e-10)
        entry[f] = float(alphas_l[nz[0]]) if len(nz) else 0.0
    first3 = sorted(entry, key=lambda k: -entry[k])[:3]
    return {'alphas_l': alphas_l, 'coefs_l': coefs_l, 'lams': lams, 'coefs_r': coefs_r, 'alpha_cv': float(cv.alpha_),
            'kept': kept, 'n_kept': len(kept), 'top': top, 'first3': first3, 'n': int(len(F)), 'p': len(FEATS)}


def fig_shrinkage(S=None, save=True):
    """Coefficient paths: ridge (left) and lasso (right) against the penalty; dashed: the lasso penalty chosen by
    walk-forward cross-validation."""
    S = S or shrinkage_paths()
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
    for j, f in enumerate(FEATS):
        col = HIGHLIGHT.get(f)
        kw = dict(color=col, lw=1.8, label=f) if col else dict(color=st.MainBlue, lw=0.6, alpha=0.35, label='_nolegend_')
        ax[0].plot(S['lams'], S['coefs_r'][j], **kw)
        ax[1].plot(S['alphas_l'], S['coefs_l'][j], **{**kw, 'label': '_nolegend_'})
    ax[0].set_xscale('log')
    ax[1].set_xscale('log')
    ax[0].set_xlabel('ridge penalty $\\lambda$')
    ax[1].set_xlabel('lasso penalty $\\lambda$')
    ax[0].set_ylabel('coefficient (GW per s.d. of the feature)')
    ax[1].axvline(S['alpha_cv'], color=st.IDAred, ls='--', lw=1)
    ax[0].set_title('ridge: all coefficients shrink smoothly', fontsize=11)
    ax[1].set_title('lasso: coefficients drop to exactly zero', fontsize=11)
    ax[1].invert_xaxis()
    ax[0].invert_xaxis()
    h0 = plt.Line2D([], [], color=st.MainBlue, lw=0.6, alpha=0.5, label='other features')
    hh, ll = ax[0].get_legend_handles_labels()
    fig.legend(hh + [h0], ll + ['other features'], loc='upper center', bbox_to_anchor=(0.5, 0.02), ncol=7, frameon=False)
    fig.tight_layout(rect=(0, 0.08, 1, 1))
    st.check_no_grey(fig)
    if save:
        st.save_fig('tsa_ch9_shrinkage')
    else:
        plt.show()
    return {k: S[k] for k in ('alpha_cv', 'kept', 'n_kept', 'top', 'first3', 'n', 'p')}


def fig_tree(y=None, end=CV_FIRST, save=True):
    """A regression tree of depth 2 for tomorrow's load (h = 1)."""
    y = load_daily() if y is None else y
    F = direct_frame(y.loc[:end], 1).dropna()
    F = F.assign(weekend=1 - F[['mon', 'tue', 'wed', 'thu', 'fri']].sum(axis=1))
    cols = ['lag0', 'same_dow', 'weekend', 'holiday', 'xmas', 'easter']
    m = DecisionTreeRegressor(max_depth=2, min_samples_leaf=20, random_state=0).fit(F[cols], F['y'])
    fig, ax = plt.subplots(figsize=(10.5, 4.4))
    plot_tree(m, feature_names=cols, filled=False, impurity=False, precision=2, fontsize=10, ax=ax, rounded=True)
    for t in ax.texts:
        t.set_color('black')
        bb = t.get_bbox_patch()
        if bb is not None:
            bb.set_edgecolor(st.MainBlue)
            bb.set_facecolor('white')
    st.check_no_grey(fig)
    if save:
        st.save_fig('tsa_ch9_tree')
    else:
        plt.show()
    tr = m.tree_
    root = {'feat': cols[tr.feature[0]], 'thr': float(tr.threshold[0]), 'n': int(tr.n_node_samples[0]),
            'value': float(tr.value[0][0][0])}
    leaves = [{'n': int(tr.n_node_samples[i]), 'value': float(tr.value[i][0][0])} for i in range(tr.node_count)
              if tr.children_left[i] == -1]
    kids = [{'feat': cols[tr.feature[i]], 'thr': float(tr.threshold[i])} for i in (tr.children_left[0], tr.children_right[0])
            if tr.children_left[i] != -1]
    r2 = float(m.score(F[cols], F['y']))
    return {'root': root, 'leaves': leaves, 'kids': kids, 'r2': r2, 'n': int(len(F))}


def ensemble_curves(y=None, split='2024-12-31', end=CV_FIRST, seed=SEED):
    """Validation MSE (January - June 2025, h = 1) of a single tree and a random forest for depth 1..14, and of
    histogram gradient boosting against the number of trees for two learning rates."""
    y = load_daily() if y is None else y
    F = direct_frame(y.loc[:end], 1).dropna()
    tr, va = F.loc[:split], F.loc[pd.Timestamp(split) + pd.Timedelta(days=1):]
    out = {'depth': list(range(1, 15)), 'tree': [], 'rf': []}
    for dep in out['depth']:
        t = DecisionTreeRegressor(max_depth=dep, random_state=seed).fit(tr[FEATS], tr['y'])
        out['tree'].append(float(np.mean((t.predict(va[FEATS]) - va['y']) ** 2)))
        f = RandomForestRegressor(n_estimators=200, max_depth=dep, max_features=0.5, n_jobs=-1, random_state=seed)
        f.fit(tr[FEATS], tr['y'])
        out['rf'].append(float(np.mean((f.predict(va[FEATS]) - va['y']) ** 2)))
    out['iters'] = list(range(1, 601))
    for lr in (0.3, 0.05):
        g = HistGradientBoostingRegressor(max_iter=600, learning_rate=lr, max_leaf_nodes=15, min_samples_leaf=20,
                                          early_stopping=False, random_state=seed).fit(tr[FEATS], tr['y'])
        out[f'gb{lr}'] = [float(np.mean((p - va['y'].values) ** 2)) for p in g.staged_predict(va[FEATS])]
    out['naive'] = float(np.mean((va['same_dow'] - va['y']) ** 2))
    out['n_tr'], out['n_va'] = int(len(tr)), int(len(va))
    return out


def fig_ensembles(E=None, save=True):
    """Left: validation MSE against depth, single tree and random forest; right: boosting against the number of trees."""
    E = E or ensemble_curves()
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.0))
    ax[0].plot(E['depth'], E['tree'], color=st.Amber, marker='o', label='single tree')
    ax[0].plot(E['depth'], E['rf'], color=st.Forest, marker='o', label='random forest (200 trees)')
    ax[0].axhline(E['naive'], color=st.Purple, ls=':', label='seasonal naive (same weekday last week)')
    ax[0].set_xlabel('maximum depth')
    ax[0].set_ylabel('validation MSE (GW$^2$)')
    ax[1].plot(E['iters'], E['gb0.3'], color=st.IDAred, label='boosting, learning rate 0.3')
    ax[1].plot(E['iters'], E['gb0.05'], color=st.MainBlue, label='boosting, learning rate 0.05')
    ax[1].axhline(E['naive'], color=st.Purple, ls=':', label='_nolegend_')
    ax[1].set_xscale('log')
    ax[1].set_xlabel('number of trees')
    ax[1].set_ylim(0, max(E['naive'], E['tree'][0]) * 1.1)
    ax[0].set_ylim(0, max(E['naive'], E['tree'][0]) * 1.1)
    st.fig_legend_bottom(fig, ncol=3, y=0.0)
    fig.tight_layout(rect=(0, 0.1, 1, 1))
    st.check_no_grey(fig)
    if save:
        st.save_fig('tsa_ch9_ensembles')
    else:
        plt.show()
    j = int(np.argmin(E['rf']))
    return {'tree_best': float(min(E['tree'])), 'tree_best_d': int(E['depth'][int(np.argmin(E['tree']))]),
            'rf_best': float(E['rf'][j]), 'rf_best_d': int(E['depth'][j]), 'naive': E['naive'],
            'gb03_min': float(min(E['gb0.3'])), 'gb03_min_it': int(np.argmin(E['gb0.3']) + 1),
            'gb03_end': float(E['gb0.3'][-1]), 'gb005_min': float(min(E['gb0.05'])),
            'gb005_min_it': int(np.argmin(E['gb0.05']) + 1), 'gb005_end': float(E['gb0.05'][-1]),
            'n_tr': E['n_tr'], 'n_va': E['n_va']}


def hicp(geo='RO'):
    """Monthly HICP index (2015 = 100) of one EU country, Eurostat prc_hicp_minr."""
    for attempt in range(4):              # the public API sometimes drops a connection: retry
        try:
            s = read_eurostat(HICP[0], HICP[1].format(geo))
            break
        except Exception:
            if attempt == 3:
                raise
            import time
            time.sleep(5)
    s.index.freq = None
    return s.rename(geo)


def fig_extrapolation(split='2020-12-01', seed=SEED, save=True):
    """Trees cannot extrapolate: one-month-ahead forecasts of the Romanian HICP index level from a random forest on
    lagged levels, a random forest on monthly log changes, and an AR(12) on log changes, all trained up to 2020."""
    p = hicp('RO').loc['2005-01-01':]
    lp = np.log(p)
    lv = pd.DataFrame({f'l{j}': p.shift(j) for j in range(1, 13)}).assign(y=p)
    ch = pd.DataFrame({f'l{j}': 100 * lp.diff().shift(j) for j in range(1, 13)}).assign(y=100 * lp.diff())
    lv, ch = lv.dropna(), ch.dropna()
    tr_l, tr_c = lv.loc[:split], ch.loc[:split]
    rf_l = RandomForestRegressor(300, min_samples_leaf=2, random_state=seed, n_jobs=-1).fit(tr_l.drop(columns='y'), tr_l['y'])
    rf_c = RandomForestRegressor(300, min_samples_leaf=2, random_state=seed, n_jobs=-1).fit(tr_c.drop(columns='y'), tr_c['y'])
    te = lv.index[lv.index > split]
    f_l = pd.Series(rf_l.predict(lv.loc[te].drop(columns='y')), index=te)
    f_c = pd.Series(p.shift(1).loc[te].values * np.exp(rf_c.predict(ch.loc[te].drop(columns='y')) / 100), index=te)
    fig, ax = plt.subplots(figsize=(10, 4.0))
    s = p.loc['2017-01-01':]
    ax.plot(s.index, s.values, color='black', lw=1.6, label='HICP index, Romania (2015 = 100)')
    ax.plot(f_l.index, f_l.values, color=st.IDAred, lw=1.6, ls='--', label='random forest on lagged levels')
    ax.plot(f_c.index, f_c.values, color=st.Forest, lw=1.6, ls='-.', label='random forest on monthly changes')
    ax.axhline(tr_l['y'].max(), color=st.Amber, ls=':', lw=1.2, label='largest level seen in training')
    ax.axvline(pd.Timestamp(split), color=st.MainBlue, ls=':', lw=1)
    ax.set_ylabel('index')
    st.legend_outside_bottom(ax, ncol=2, y=-0.15)
    st.check_no_grey(fig)
    if save:
        st.save_fig('tsa_ch9_extrapolation')
    else:
        plt.show()
    e_l = (p.loc[te] - f_l)
    e_c = (p.loc[te] - f_c)
    return {'max_train': float(tr_l['y'].max()), 'last': float(p.iloc[-1]), 'last_d': str(p.index[-1].date()),
            'f_l_last': float(f_l.iloc[-1]), 'f_l_max': float(f_l.max()), 'rmse_l': float(np.sqrt(np.mean(e_l ** 2))),
            'rmse_c': float(np.sqrt(np.mean(e_c ** 2))), 'n_te': int(len(te))}

## Run

In [ ]:
print(fig_shrinkage())
print(fig_tree())
print(fig_ensembles())
print(fig_extrapolation())

![tsa_ch9_shrinkage](./tsa_ch9_shrinkage.png)

Output: `tsa_ch9_shrinkage.pdf`

![tsa_ch9_tree](./tsa_ch9_tree.png)

Output: `tsa_ch9_tree.pdf`

![tsa_ch9_ensembles](./tsa_ch9_ensembles.png)

Output: `tsa_ch9_ensembles.pdf`

![tsa_ch9_extrapolation](./tsa_ch9_extrapolation.png)

Output: `tsa_ch9_extrapolation.pdf`